In [1]:
import numpy as np
import random

In [7]:
class TicTacToe:
    def __init__(self):
        self.board = [' '] * 9

    def reset(self):
        self.board = [' '] * 9
        return tuple(self.board)

    def available_actions(self):
        return [i for i in range(9) if self.board[i] == ' ']

    def make_move(self, action, player):
        self.board[action] = player

    def check_winner(self):
        wins = [
            (0,1,2), (3,4,5), (6,7,8),
            (0,3,6), (1,4,7), (2,5,8),
            (0,4,8), (2,4,6)
        ]

        for a, b, c in wins:
            if self.board[a] == self.board[b] == self.board[c] != ' ':
                return self.board[a]

        if ' ' not in self.board:
            return 'Draw'

        return None

    def display(self):
        for i in range(0, 9, 3):
            print(self.board[i:i+3])
        print()

In [8]:
game = TicTacToe()
game.display()

[' ', ' ', ' ']
[' ', ' ', ' ']
[' ', ' ', ' ']



In [9]:
q_table = {}

alpha = 0.1
gamma = 0.9
epsilon = 0.2

In [10]:
def get_q(state, action):
    return q_table.get((state, action), 0.0)

In [11]:
def choose_action(state, actions):
    if random.random() < epsilon:
        return random.choice(actions)

    return max(actions, key=lambda a: get_q(state, a))

In [12]:
def update_q(state, action, reward, next_state, next_actions):
    old_q = get_q(state, action)

    if next_actions:
        future_q = max(get_q(next_state, a) for a in next_actions)
    else:
        future_q = 0

    q_table[(state, action)] = old_q + alpha * (
        reward + gamma * future_q - old_q
    )

In [13]:
game = TicTacToe()

for episode in range(10000):
    state = game.reset()

    while True:
        actions = game.available_actions()
        action = choose_action(state, actions)

        game.make_move(action, 'X')
        result = game.check_winner()

        if result:
            reward = 1 if result == 'X' else 0
            update_q(state, action, reward, None, [])
            break

        opponent_action = random.choice(game.available_actions())
        game.make_move(opponent_action, 'O')

        result = game.check_winner()

        if result:
            reward = -1 if result == 'O' else 0
            update_q(state, action, reward, None, [])
            break

        next_state = tuple(game.board)
        next_actions = game.available_actions()

        update_q(state, action, 0, next_state, next_actions)
        state = next_state

print("Training Completed!")
print("Q-table entries:", len(q_table))

Training Completed!
Q-table entries: 2421


In [14]:
def play_game():
    game = TicTacToe()
    state = game.reset()

    while True:
        actions = game.available_actions()
        action = max(actions, key=lambda a: get_q(state, a))

        game.make_move(action, 'X')
        print("AI Move:")
        game.display()

        result = game.check_winner()

        if result:
            print("Result:", result)
            break

        opponent_action = random.choice(game.available_actions())
        game.make_move(opponent_action, 'O')

        print("Opponent Move:")
        game.display()

        result = game.check_winner()

        if result:
            print("Result:", result)
            break

In [15]:
play_game()

AI Move:
['X', ' ', ' ']
[' ', ' ', ' ']
[' ', ' ', ' ']

Opponent Move:
['X', ' ', ' ']
[' ', 'O', ' ']
[' ', ' ', ' ']

AI Move:
['X', 'X', ' ']
[' ', 'O', ' ']
[' ', ' ', ' ']

Opponent Move:
['X', 'X', ' ']
['O', 'O', ' ']
[' ', ' ', ' ']

AI Move:
['X', 'X', 'X']
['O', 'O', ' ']
[' ', ' ', ' ']

Result: X
